In [ ]:
import pandas as pd
from tqdm import tqdm

### Getting pdf of all documents:

! this extracts only ESDS documents with language set to English (but it is erroneously set to all documents)

```
python scrape_eib_esds.py 
```

### Extract text from pdfs

```
python extract_corpus.py --year-from 2014 --year-to 2026 --output-dir all
```

This will create `documents.parquet` and `sentences.parquet`

### Lemmatization, POS-tagging, syntactic parsing

see the model card for requirements

```
curl -sSfL https://hf.co/git-xet/install.sh | sh
git clone https://huggingface.co/HPLT/hplt_gpt_bert_base_3_0_eng_Latn-UD
cp ud_parsing.py ud_parsing.sh hplt_gpt_bert_base_3_0_eng_Latn-UD/
cd hplt_gpt_bert_base_3_0_eng_Latn-UD/
sbatch ud_parsing.sh
```

! I am not sure git-xet is available on olivia, but I did not check, I just scp-ed the model repo from my laptop hehe
! I fixed the mess about to.device but the model repos on HF are not updated yet

This will create sentences_parsed.parquet, nouns.txt and noun_phrases.txt

In [ ]:
doc_id = pd.read_parquet("all/documents.parquet")

In [ ]:
doc_id.shape

In [ ]:
doc_id.head()

In [ ]:
doc_id[doc_id.title.str.contains("version")]

In [ ]:
doc_id.iloc[1770].title

### Document-level language identification

In [ ]:
# pip install fasttext==0.9.3 huggingface-hub==0.35.3 numpy==1.23.5 regex==2024.4.28
import fasttext
from huggingface_hub import hf_hub_download
import regex

In [ ]:
# defines what we want to remove from string for langID
NONWORD_REPLACE_STR = r"[^\p{Word}\p{Zs}]|\d"  # either (not a word nor a space) or (is digit)
NONWORD_REPLACE_PATTERN = regex.compile(NONWORD_REPLACE_STR)
SPACE_PATTERN = regex.compile(r"\s\s+")  # squeezes sequential whitespace


def preprocess(text):
    text = text.strip().replace('\n', ' ').lower()
    text = regex.sub(SPACE_PATTERN, " ", text)
    text = regex.sub(NONWORD_REPLACE_PATTERN, "", text)
    return text

model_path = hf_hub_download(
    repo_id="HPLT/OpenLID-v3", filename="openlid-v3.bin",
) # may take some time
model = fasttext.load_model(model_path)
text = "Maskinsjefen er oppteken av å løfta fram dei maritime utdanningane."
text = preprocess(text)
print(
      model.predict(
          text=text,
          k=1,
          threshold=0.5,
          on_unicode_error="strict",
      ),
)
# should output: (('__label__nno_Latn',), array([0.99999893]))

In [ ]:
tqdm.pandas()

In [ ]:
def predict_lang(x):
    lang = model.predict(
          text=preprocess(x),
          k=1,
          threshold=0.5,
          on_unicode_error="strict",
      )[0]
    if lang:
        return lang[0].removeprefix('__label__')
    else:
        return ""
    #return lang # if we wanted to check if there are no more than 1 prediction per document, but we have no such

In [ ]:
doc_id['lang'] = doc_id.text.progress_apply(predict_lang)

In [ ]:
doc_id.head()

In [ ]:
doc_id.lang.unique()

In [ ]:
eng = doc_id[~doc_id.lang.isin({'fra_Latn', 'ell_Grek', 'als_Latn'})]
eng.shape

In [ ]:
doc_id[doc_id.lang=="ell_Grek"]

In [ ]:
doc_id[doc_id.lang=="fra_Latn"]

In [ ]:
doc_id[doc_id.lang=="als_Latn"]

### Sentence-level language identification

In [ ]:
data = pd.read_parquet("all/sentences_parsed.parquet")
data.head()

In [ ]:
data['lang'] = data.sentence.progress_apply(predict_lang)

In [ ]:
data['lang'].unique()

In [ ]:
data[data.lang=='ekk_Latn']

In [ ]:
data.iloc[43694].sentence

In [ ]:
data.iloc[142090].sentence

In [ ]:
data.head()

In [ ]:
data.shape

### Removing sentences from non-english documents

In [ ]:
eng_sent = data[data.doc_id.isin(eng.doc_id)]

In [ ]:
eng_sent.shape